# SU(2) Hamiltonian Demo

A compact demo notebook for the current story:

```text
Hamiltonian H and evolution time t
    -> SU(2) diffusion-generated local gates
    -> fixed 3-qubit CZ template search
    -> local SU(2) refinement
    -> concrete circuit decomposition for exp(-i H t)
```

The first two examples are deliberately simple (`XII` and `YII`) so the answer can be checked by hand. The final example is a small transverse-Ising-style 3-qubit Hamiltonian.

In [ ]:
BRANCH = "main"
!pip install -q --force-reinstall --no-deps git+https://github.com/joe-singh/su2diffusion.git@{BRANCH}

In [ ]:
from dataclasses import replace

import torch

from su2diffusion import (
    center_names_for_config,
    compose_three_qubit_template_units,
    get_experiment_config,
    make_hamiltonian_target,
    pauli_string_matrix,
    plot_hamiltonian_demo,
    print_hamiltonian_demo,
    quaternion_to_unitary,
    run_experiment,
    run_three_qubit_hamiltonian_demo,
    unitary_fidelity,
)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

## Train the local SU(2) gate generator

This trains the conditional single-gate diffusion model on Clifford neighborhoods and gives us a pool of generated local gates. The rest of the notebook reuses this pool.

In [ ]:
CONFIG_NAME = "baseline-clifford-cond"
EVAL_COUNT = 1000

config = get_experiment_config(CONFIG_NAME)
config = replace(config, sample_count=EVAL_COUNT, reference_count=EVAL_COUNT)

result = run_experiment(config, device=device)

center_names = center_names_for_config(result.config.data)
local_gates = result.generated_stochastic
local_labels = [center_names[int(label)] for label in result.stochastic_labels]

print(f"trained config: {result.config.name}")
print(f"generated local gates: {tuple(local_gates.shape)}")

## Demo helpers

`run_demo` searches over generated local gates inside the fixed 3-qubit template

```text
L0 - CZ01 - L1 - CZ12 - L2 - CZ01 - L3 - CZ12 - L4
```

then refines the selected local `SU(2)` gates while keeping the CZ skeleton fixed.

In [ ]:
def run_demo(target, *, seed=9917, n_random_candidates=10_000):
    return run_three_qubit_hamiltonian_demo(
        target,
        generated_gates=local_gates,
        generated_labels=local_labels,
        token_stacks=None,
        template="line-4cz",
        source="generated-search",
        n_random_candidates=n_random_candidates,
        top_k=5,
        seed=seed,
        refinement_steps=80,
        refinement_lr=0.05,
        threshold=0.99,
    )


def align_global_phase(candidate, target):
    overlap = torch.trace(target.conj().T @ candidate)
    phase = overlap / overlap.abs().clamp_min(1e-12)
    return candidate / phase


def obvious_single_qubit_stack(demo, axis, angle):
    stack = torch.zeros_like(demo.refinement.refined_gates)
    stack[:, 0] = 1.0
    half_angle = torch.tensor(angle / 2, dtype=stack.dtype, device=stack.device)
    zero = torch.zeros((), dtype=stack.dtype, device=stack.device)
    if axis == "X":
        stack[0] = torch.stack([torch.cos(half_angle), torch.sin(half_angle), zero, zero])
    elif axis == "Y":
        stack[0] = torch.stack([torch.cos(half_angle), zero, torch.sin(half_angle), zero])
    elif axis == "Z":
        stack[0] = torch.stack([torch.cos(half_angle), zero, zero, torch.sin(half_angle)])
    else:
        raise ValueError(f"Unknown axis {axis!r}")
    return stack


def check_single_pauli_demo(demo, pauli_string, coefficient, time, axis_label):
    U_circuit = compose_three_qubit_template_units(
        quaternion_to_unitary(demo.refinement.refined_gates),
        demo.template,
    )
    pauli = pauli_string_matrix(pauli_string, n_qubits=3, device=U_circuit.device)
    U_manual = torch.linalg.matrix_exp(-1j * coefficient * time * pauli)

    obvious_stack = obvious_single_qubit_stack(demo, axis_label, angle=2 * coefficient * time)
    U_obvious = compose_three_qubit_template_units(
        quaternion_to_unitary(obvious_stack),
        demo.template,
    )

    print(f"\nfidelity to manual exp(-i {coefficient * time:g} {pauli_string}):")
    print("  generated/refined circuit:", unitary_fidelity(U_circuit, U_manual))
    print(f"  obvious R{axis_label.lower()}-on-q0 circuit:", unitary_fidelity(U_obvious, U_manual))

    U_aligned = align_global_phase(U_circuit, U_manual)
    print("\nmax generated-circuit matrix entry error after phase alignment:")
    print((U_aligned - U_manual).abs().max().item())

    ket000 = torch.zeros(8, dtype=torch.complex64, device=U_circuit.device)
    ket000[0] = 1.0

    out_manual = U_manual @ ket000
    out_circuit = U_aligned @ ket000
    out_obvious = U_obvious @ ket000

    print(f"\nExpected nonzero amplitudes for exp(-i {coefficient * time:g} {axis_label}) on q0:")
    print("|000>:", out_manual[0])
    print("|100>:", out_manual[4])

    print("\nGenerated/refined circuit amplitudes:")
    print("|000>:", out_circuit[0])
    print("|100>:", out_circuit[4])

    print(f"\nObvious R{axis_label.lower()}-on-q0 circuit amplitudes:")
    print("|000>:", out_obvious[0])
    print("|100>:", out_obvious[4])

    print("\nGenerated output-state max error:")
    print((out_circuit - out_manual).abs().max().item())


def summarize_demos(demos):
    header = "case                 proposal   refined   steps>=0.99   mean move   max move"
    print(header)
    print("-" * len(header))
    for name, demo in demos:
        steps = str(demo.steps_to_threshold) if demo.steps_to_threshold >= 0 else "miss"
        print(
            f"{name:<20} "
            f"{demo.refinement.initial_fidelity:>8.4f} "
            f"{demo.refinement.refined_fidelity:>9.4f} "
            f"{steps:>11} "
            f"{demo.movement_mean:>11.4f} "
            f"{demo.movement_max:>10.4f}"
        )

## Sanity check: `H = 0.5 XII`, `t = 1`

This target is just an `X` rotation on qubit 0. The obvious circuit is one local `Rx(1.0)` gate on `q0` and identities everywhere else. The generated/refined circuit may look less obvious, but it should match the same unitary.

In [ ]:
simple_x_target = make_hamiltonian_target(
    [("XII", 0.5)],
    time=1.0,
    name="simple-X-on-q0",
    n_qubits=3,
    device=device,
)

simple_x_demo = run_demo(simple_x_target, seed=9917)

print_hamiltonian_demo(simple_x_demo)
plot_hamiltonian_demo(simple_x_demo)
check_single_pauli_demo(simple_x_demo, "XII", coefficient=0.5, time=1.0, axis_label="X")

## Sanity check: `H = 0.5 YII`, `t = 1`

Same idea, but now the obvious single-qubit solution is `Ry(1.0)` on qubit 0.

In [ ]:
simple_y_target = make_hamiltonian_target(
    [("YII", 0.5)],
    time=1.0,
    name="simple-Y-on-q0",
    n_qubits=3,
    device=device,
)

simple_y_demo = run_demo(simple_y_target, seed=9927)

print_hamiltonian_demo(simple_y_demo)
plot_hamiltonian_demo(simple_y_demo)
check_single_pauli_demo(simple_y_demo, "YII", coefficient=0.5, time=1.0, axis_label="Y")

## Nontrivial demo: transverse-Ising-style Hamiltonian

Now the target has commuting `ZZ` couplings plus local transverse `X` fields:

```text
H = 0.35 ZZI + 0.35 IZZ + 0.20 XII + 0.20 IXI + 0.20 IIX
```

This is no longer a single obvious local rotation. The point of the report is to show the proposed circuit, how quickly local `SU(2)` refinement reaches high fidelity, and how much each local slot had to move.

In [ ]:
ising_target = make_hamiltonian_target(
    [
        ("ZZI", 0.35),
        ("IZZ", 0.35),
        ("XII", 0.20),
        ("IXI", 0.20),
        ("IIX", 0.20),
    ],
    time=0.8,
    name="three-qubit-transverse-ising",
    n_qubits=3,
    device=device,
)

ising_demo = run_demo(ising_target, seed=9937, n_random_candidates=20_000)

print_hamiltonian_demo(ising_demo)
plot_hamiltonian_demo(ising_demo)

## One-line summary

The simple cases are there for trust. The Ising case is the first “show this to someone” Hamiltonian-to-circuit example.

In [ ]:
summarize_demos([
    ("XII sanity", simple_x_demo),
    ("YII sanity", simple_y_demo),
    ("transverse Ising", ising_demo),
])